In [27]:

import argparse

from qiskit_aer import AerSimulator

from src.deutsch_jozsa import (
    balanced_oracle,
    constant_oracle,
    deutsch_jozsa_circuit,
    interpret_counts,
)


In [28]:
n_bits = 3
n_shots = 100
oracle_type = "contant" #Balanced or anyother thing
oracle = (
        constant_oracle(n_bits)
        if oracle_type == "constant"
        else balanced_oracle(n_bits)
    )
qc = deutsch_jozsa_circuit(oracle, n_bits)
print(qc.draw(output="text"))

     ┌───┐      ░                 ░ ┌───┐┌─┐      
q_0: ┤ H ├──────░───■─────────────░─┤ H ├┤M├──────
     ├───┤      ░   │             ░ ├───┤└╥┘┌─┐   
q_1: ┤ H ├──────░───┼────■────────░─┤ H ├─╫─┤M├───
     ├───┤      ░   │    │        ░ ├───┤ ║ └╥┘┌─┐
q_2: ┤ H ├──────░───┼────┼────■───░─┤ H ├─╫──╫─┤M├
     ├───┤┌───┐ ░ ┌─┴─┐┌─┴─┐┌─┴─┐ ░ └───┘ ║  ║ └╥┘
q_3: ┤ X ├┤ H ├─░─┤ X ├┤ X ├┤ X ├─░───────╫──╫──╫─
     └───┘└───┘ ░ └───┘└───┘└───┘ ░       ║  ║  ║ 
c: 3/═════════════════════════════════════╩══╩══╩═
                                          0  1  2 


# Simulatore NOISELESS

Primo giro su simulatore ideale senza rumore e coerenza perfetta.

In [29]:

sim = AerSimulator()
result = sim.run(qc, shots=n_shots).result()
counts = result.get_counts()
verdict = interpret_counts(counts)
print(f"\nCounts: {counts}")
print(f"Verdetto: {verdict.upper()}")


Counts: {'111': 100}
Verdetto: BALANCED


In [30]:
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

# 1. Definisci l'errore (es. 5% di probabilità di errore depolarizzante su gate a 1 qubit)
prob_1 = 0.05
error_1q = depolarizing_error(prob_1, 1)

# 2. Crea il modello di rumore e aggiungi l'errore ai gate specifici
noise_model = NoiseModel()
# Applica l'errore ai gate (es. applicalo a tutti i gate Pauli-X e Hadamard)
noise_model.add_all_qubit_quantum_error(error_1q, ['x', 'h'])

# 3. Inizializza il simulatore passandogli il modello di rumore
sim_noise = AerSimulator(noise_model=noise_model)

# 4. Esegui il circuito come facevi prima
result = sim_noise.run(qc, shots=100).result()
counts = result.get_counts()
verdict = interpret_counts(counts)

print(f"\nCounts con rumore personalizzato: {counts}")
print(f"Verdetto: {verdict.upper()}")


Counts con rumore personalizzato: {'011': 3, '000': 2, '101': 6, '110': 8, '111': 81}
Verdetto: BALANCED


In [31]:
from dotenv import load_dotenv
import os
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler


In [ ]:
def get_service() -> QiskitRuntimeService:
    """Crea il service IBM dal token nel file .env."""
    load_dotenv()
    token = os.getenv("IBM_QUANTUM_TOKEN")
    if not token or token == "your_token_here":
        raise SystemExit(
            "Token IBM mancante. Copia .env.example in .env e inserisci "
            "IBM_QUANTUM_TOKEN."
        )
    channel = os.getenv("IBM_CHANNEL", "ibm_quantum_platform")
    instance = None
    print(token)
    return QiskitRuntimeService(channel="ibm_quantum_platform", token=token, instance=instance)



In [44]:
service = get_service()
#scelgo di default il least-busy
backend = service.least_busy(operational=True, simulator=False)
print(f"Backend selezionato: {backend.name}")

oracle = (
        constant_oracle(n_bits)
        if oracle_type == "constant"
        else balanced_oracle(n_bits)
    )
qc = deutsch_jozsa_circuit(oracle, n_bits)

    # Transpilazione per il backend reale
pm = generate_preset_pass_manager(optimization_level=1, backend=backend, translation_method="translator")
isa_circuit = pm.run(qc)

sampler = Sampler(mode=backend)
job = sampler.run([isa_circuit], shots=n_shots)
print(f"Job ID: {job.job_id()}  -- in attesa dei risultati...")

result = job.result()
counts = result[0].data.c.get_counts()

verdict = interpret_counts(counts)
print(f"\nCounts: {counts}")
print(f"Verdetto: {verdict.upper()}")


qiskit_runtime_service._discover_account:WARNING:2026-06-26 19:51:12,222: Loading account with the given token. A saved account will not be used.


1_f2MjzPPe5QIA0eDdcVaEeVkYk0hUnJC2hMM584BJG2


qiskit_runtime_service.__init__:WARNING:2026-06-26 19:51:14,830: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-06-26 19:51:15,332: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-06-26 19:51:18,685: Using instance: open-instance, plan: open


Backend selezionato: ibm_kingston
Job ID: d8vbo66mvj5c73ehf1pg  -- in attesa dei risultati...

Counts: {'111': 93, '001': 1, '011': 2, '101': 1, '100': 1, '110': 2}
Verdetto: BALANCED
